# Interactive Electrode Explorer

Load a Maxwell recording, visually select electrodes on the MEA chip layout, and plot raw, LFP (0.5-300 Hz), and spike (300-3000 Hz) filtered traces with color coding and offsets.

In [ ]:
%matplotlib widget
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import ipywidgets as widgets
from IPython.display import display
from matplotlib.widgets import RectangleSelector

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
src_path = project_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from acute_slice_mea.recording import load_maxwell_recording, prepare_recordings
from acute_slice_mea.electrodes import build_electrode_table
from acute_slice_mea.spectral import get_traces_safe

## Configuration

In [ ]:
data_path = "/mnt/benshalom-nas/raw_data/rbs_maxtwo_desktop/harddisk24tbvol1/KCNT1_MEASlices_06062025_CA_PS/250606/M07896/Network/000003/data.raw.h5"
well_id = "well004"

lfp_low_hz = 0.5
lfp_high_hz = 300
spike_low_hz = 300
spike_high_hz = 3000

trace_start_sec = 0
trace_duration_sec = 10

## Load and Prepare Recording

In [ ]:
print("Loading Maxwell recording...")
raw = load_maxwell_recording(data_path, well_id)
print(f"Loaded {len(raw.get_channel_ids())} channels")

print("Preparing filtered recordings...")
recordings = prepare_recordings(
    raw,
    lfp_low_hz=lfp_low_hz,
    lfp_high_hz=lfp_high_hz,
    spike_low_hz=spike_low_hz,
    spike_high_hz=spike_high_hz,
)

print("Building electrode table...")
electrodes = build_electrode_table(raw.get_probe(), raw)
recorded_electrodes = electrodes[electrodes["recorded"].astype(bool)].copy()
print(f"Found {len(recorded_electrodes)} recorded electrodes")

fs = float(raw.get_sampling_frequency())
print(f"Sampling frequency: {fs} Hz")

# Prepare data for probe map
contact_positions = electrodes[["x_um", "y_um"]].values
recorded_electrode_ids = recorded_electrodes["electrode_id"].astype(int).values
recorded_positions = contact_positions[recorded_electrode_ids]
electrode_to_channel_map = dict(zip(electrodes["electrode_id"].astype(int), electrodes["channel_id"]))
recorded_lookup = {int(eid): electrode_to_channel_map[eid] for eid in recorded_electrode_ids}

## Interactive MEA Electrode Selector

In [ ]:
# Global state for selected electrodes
selected_electrode_ids = []

def _selection_table(electrode_ids):
    rows = []
    for eid in electrode_ids:
        eid = int(eid)
        ch_id = recorded_lookup.get(eid)
        if ch_id is None:
            continue
        x_um, y_um = contact_positions[eid]
        rows.append({
            'electrode_id': eid,
            'channel_id': ch_id,
            'x_um': float(x_um),
            'y_um': float(y_um),
        })
    return pd.DataFrame(rows, columns=['electrode_id', 'channel_id', 'x_um', 'y_um'])

def set_selected_electrodes(electrode_ids, append=False):
    """Set or append selected recorded electrodes and update display."""
    global selected_electrode_ids
    
    valid = []
    invalid = []
    for eid in electrode_ids:
        try:
            eid = int(eid)
        except (TypeError, ValueError):
            invalid.append(eid)
            continue
        if eid in recorded_lookup:
            valid.append(eid)
        else:
            invalid.append(eid)
    
    if append:
        valid = selected_electrode_ids + valid
    
    selected_electrode_ids = sorted(dict.fromkeys(valid))
    _refresh_selector_plot()
    _print_selection_status(invalid)
    return _selection_table(selected_electrode_ids)

def clear_selected_electrodes(_=None):
    return set_selected_electrodes([], append=False)

def _refresh_selector_plot():
    if len(selected_electrode_ids) == 0:
        selected_plot.set_offsets(np.empty((0, 2)))
    else:
        selected_plot.set_offsets(contact_positions[selected_electrode_ids])
    fig.canvas.draw_idle()

def _print_selection_status(invalid=None):
    invalid = invalid or []
    with selector_output:
        selector_output.clear_output()
        print(f"Selected {len(selected_electrode_ids)} recorded electrode(s).")
        if selected_electrode_ids:
            display(_selection_table(selected_electrode_ids))
        if invalid:
            print(f"Ignored unrecorded/invalid electrode IDs: {invalid}")

def _parse_electrode_text(text):
    tokens = text.replace('\n', ',').replace(';', ',').replace(' ', ',').split(',')
    return [token.strip() for token in tokens if token.strip()]

def apply_manual_selection(_=None):
    return set_selected_electrodes(_parse_electrode_text(manual_electrode_ids.value), append=False)

def append_manual_selection(_=None):
    return set_selected_electrodes(_parse_electrode_text(manual_electrode_ids.value), append=True)

def _toggle_electrode(eid):
    current = set(selected_electrode_ids)
    eid = int(eid)
    if eid in current:
        current.remove(eid)
    else:
        current.add(eid)
    return set_selected_electrodes(sorted(current), append=False)

def on_pick(event):
    if event.artist is not recorded_scatter or len(event.ind) == 0:
        return
    mouse_xy = np.array([event.mouseevent.xdata, event.mouseevent.ydata], dtype=float)
    candidate_positions = recorded_positions[event.ind]
    nearest_local_idx = np.argmin(np.sum((candidate_positions - mouse_xy) ** 2, axis=1))
    eid = recorded_electrode_ids[event.ind[nearest_local_idx]]
    _toggle_electrode(eid)

def on_rectangle_select(eclick, erelease):
    if eclick.xdata is None or erelease.xdata is None:
        return
    x0, x1 = sorted([eclick.xdata, erelease.xdata])
    y0, y1 = sorted([eclick.ydata, erelease.ydata])
    mask = (
        (recorded_positions[:, 0] >= x0) & (recorded_positions[:, 0] <= x1) &
        (recorded_positions[:, 1] >= y0) & (recorded_positions[:, 1] <= y1)
    )
    set_selected_electrodes(recorded_electrode_ids[mask], append=True)

# Create the probe map figure
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(contact_positions[:, 0], contact_positions[:, 1], s=20, c='lightgray', marker='s', label='All probe electrodes')
recorded_scatter = ax.scatter(
    recorded_positions[:, 0],
    recorded_positions[:, 1],
    s=70,
    c='tomato',
    marker='o',
    label='Recorded electrodes',
    picker=True,
    pickradius=5,
)
selected_plot = ax.scatter(
    [],
    [],
    s=100,
    facecolors='none',
    edgecolors='royalblue',
    linewidths=2.0,
    label='Selected electrodes',
)
ax.set_title('Interactive MEA Electrode Selector\nClick a recorded electrode to toggle; drag a rectangle to add multiple', fontsize=12)
ax.set_xlabel('X Position (μm)')
ax.set_ylabel('Y Position (μm)')
ax.set_aspect('equal', adjustable='box')
ax.legend(loc='best')
ax.grid(alpha=0.2)

rectangle_selector = RectangleSelector(
    ax,
    on_rectangle_select,
    useblit=True,
    button=[1],
    interactive=True,
    props=dict(facecolor='royalblue', edgecolor='royalblue', alpha=0.15, fill=True),
)
fig.canvas.mpl_connect('pick_event', on_pick)

# Control widgets
manual_electrode_ids = widgets.Text(
    value='',
    placeholder='Example: 500, 501, 520',
    description='Electrodes:',
    layout=widgets.Layout(width='400px'),
)
set_button = widgets.Button(description='Set from text', button_style='primary')
append_button = widgets.Button(description='Append from text')
clear_button = widgets.Button(description='Clear selection', button_style='warning')
selector_output = widgets.Output()

set_button.on_click(apply_manual_selection)
append_button.on_click(append_manual_selection)
clear_button.on_click(clear_selected_electrodes)

display(widgets.HBox([manual_electrode_ids, set_button, append_button, clear_button]))
display(selector_output)
_print_selection_status()

## Plot Traces by Signal Type

In [ ]:
def plot_traces(_=None):
    """Plot raw, LFP, and spike traces for selected electrodes in three separate figures."""
    plot_output.clear_output(wait=True)
    with plot_output:
        if not selected_electrode_ids:
            print("No electrodes selected. Use the probe map above to select electrodes.")
            return
        
        eids = list(selected_electrode_ids)
        channel_ids = [recorded_lookup[eid] for eid in eids]
        start_sec = start_time_slider.value
        
        # Load traces for the selected time window
        start_frame = int(start_sec * fs)
        end_frame = int((start_sec + trace_duration_sec) * fs)
        end_frame = min(end_frame, raw.get_num_samples())
        
        raw_traces = get_traces_safe(recordings["raw"], start_frame, end_frame, channel_ids, return_scaled=True)
        lfp_traces = get_traces_safe(recordings["lfp"], start_frame, end_frame, channel_ids, return_scaled=True)
        spike_traces = get_traces_safe(recordings["spike"], start_frame, end_frame, channel_ids, return_scaled=True)
        
        time_sec = np.arange(raw_traces.shape[0]) / fs + start_sec
        
        # Color palette: cycle through tab10 colors
        colors = [cm.tab10(i % 10) for i in range(len(eids))]
        
        # Plot three figures: one per signal type
        for sig_name, traces in [("RAW", raw_traces), 
                                   (f"LFP ({lfp_low_hz}–{lfp_high_hz} Hz)", lfp_traces), 
                                   (f"SPIKE ({spike_low_hz}–{spike_high_hz} Hz)", spike_traces)]:
            # Compute offset step based on signal amplitude
            scale = np.nanpercentile(np.abs(traces), 95) or 1.0
            offset_step = scale * 3.0
            
            # Create figure
            fig, ax = plt.subplots(figsize=(14, max(5, len(eids) * 0.8)))
            ax.set_title(
                f"{sig_name} — {len(eids)} electrode(s), t={start_sec:.1f}–{start_sec + trace_duration_sec:.1f}s",
                fontsize=12,
                fontweight='bold'
            )
            
            # Plot each electrode's trace
            for col_idx, (eid, color) in enumerate(zip(eids, colors)):
                offset = col_idx * offset_step
                ax.plot(
                    time_sec,
                    traces[:, col_idx] + offset,
                    color=color,
                    linewidth=0.8,
                    label=f"E{eid}"
                )
            
            # Set y-axis ticks and labels to electrode IDs at their offsets
            ytick_pos = [i * offset_step for i in range(len(eids))]
            ax.set_yticks(ytick_pos)
            ax.set_yticklabels([f"E{eid}" for eid in eids], fontsize=9)
            
            ax.set_xlabel("Time (s)", fontsize=11)
            ax.set_ylabel("Electrode ID (offset)", fontsize=11)
            ax.legend(loc='upper right', fontsize=8, ncol=max(1, len(eids) // 8))
            ax.grid(alpha=0.3)
            
            plt.tight_layout()
            plt.show()

start_time_slider = widgets.FloatSlider(
    value=trace_start_sec,
    min=0,
    max=max(0.1, (raw.get_num_samples() / fs) - trace_duration_sec),
    step=1,
    description="Start time (s):",
    layout=widgets.Layout(width='400px'),
)

plot_button = widgets.Button(
    description="Plot Traces",
    button_style='success',
    tooltip="Plot raw, LFP, and spike traces for selected electrodes",
)
plot_output = widgets.Output()

plot_button.on_click(plot_traces)

display(widgets.HBox([start_time_slider, plot_button]))
display(plot_output)